In [ ]:
"""
===============================================================================
EXCESS HEAT FACTOR (EHF) BASED ANALYSIS OF HEATWAVES IN GHANA
===============================================================================

Companion code to:
    Yamba, E. I., Amponsah, D., Pinkrah, N. O., Oppong, S., Adjei, M. J.,
    Ablerdu, A. K., Wemegah, C. S. (2026).
    "Excess Heat Factor Based Analysis of Heatwave Prevalence, Frequency,
     Duration and Intensity in Ghana."

The scripts below are arranged in the order the corresponding results appear
in the paper (Sections 3.1 -> 3.4):

    1. Standardized Temperature Anomaly Index (STAI)     -> Fig. 2
    2. Heatwave detection via the Excess Heat Factor     -> Figs. 3-7
    3. Total heatwave days and 5-year rolling mean       -> Fig. 8
    4. Station-level trends in annual heatwave days      -> Table 3
    5. Return levels of EHF intensity and duration       -> Fig. 9
       (Peaks-Over-Threshold with the Generalized Pareto Distribution)

Data source: Ghana Meteorological Agency (GMet), 1960-2022, gap-filled
             with ERA5 reanalysis.

"""


# %% ==========================================================================
# 1. STANDARDIZED TEMPERATURE ANOMALY INDEX (STAI)
#    Paper: Section 3.1  --  Trends in annual Tmax, Tmin, and Tmean
#    Output: Fig. 2  --  Standardized_Temperature_Anomalies_Trends_Zones.png
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import pymannkendall as mk
# === 1. FILES AND ZONE NAMES ===
stations = {
    "Navrongo": "Sudan Savannah",
    "Tamale": "Guinea Savannah",
    "Sunyani": "Transition",
    "Kumasi": "Forest",
    "Accra": "Coastal"
}
# === 2. CREATE SUBPLOTS FOR EACH ZONE ===
fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(12, 18), sharex=False)
axes = axes.flatten()
for i, (name, zone) in enumerate(stations.items()):
    print(f"Processing {name} ({zone})...")
    # --- Load Tmax and Tmin ---
    Tx = pd.read_csv(f"{name}_1960_2022_dTx_complete.txt", sep=r"\s+", header=None,
                     names=["year", "month", "day", "Tmax"])
    Tn = pd.read_csv(f"{name}_1960_2022_dTn_complete.txt", sep=r"\s+", header=None,
                     names=["year", "month", "day", "Tmin"])
    # Drop missing values
    Tx.dropna(subset=["Tmax"], inplace=True)
    Tn.dropna(subset=["Tmin"], inplace=True)
    # Compute annual means
    annual_tx = Tx.groupby("year")["Tmax"].mean().reset_index()
    annual_tn = Tn.groupby("year")["Tmin"].mean().reset_index()
    # Merge Tmax and Tmin to compute Tmean
    merged = pd.merge(annual_tx, annual_tn, on="year", how="inner")
    merged["Tmean"] = (merged["Tmax"] + merged["Tmin"]) / 2
    # Compute standardized anomalies for each variable
    for var in ["Tmax", "Tmin", "Tmean"]:
        mean_val = merged[var].mean()
        std_val = merged[var].std()
        merged[f"{var}_anomaly"] = (merged[var] - mean_val) / std_val
        merged[f"{var}_smooth"] = merged[f"{var}_anomaly"].rolling(window=5, center=True).mean()
    # === 3. TREND ANALYSIS (Mann-Kendall + Sen's slope in °C/decade) ===
    trend_results = {}
    for var in ["Tmax", "Tmin", "Tmean"]:
        result = mk.original_test(merged[var].dropna())
        slope_decade = result.slope * 10  # Convert °C/year → °C/decade
        p_value = result.p
        trend_results[var] = (slope_decade, p_value)
    # === 4. PLOT FOR THIS ZONE ===
    ax = axes[i]
    # Retrieve trend info for legends
    Tmax_slope, Tmax_p = trend_results["Tmax"]
    Tmin_slope, Tmin_p = trend_results["Tmin"]
    Tmean_slope, Tmean_p = trend_results["Tmean"]
    # Round values neatly
    Tmax_label = f"Tmax Anomaly "#(Slope={Tmax_slope:.3f}°C/dec, p={Tmax_p:.3f})"
    Tmin_label = f"Tmin Anomaly "#(Slope={Tmin_slope:.3f}°C/dec, p={Tmin_p:.3f})"
    Tmean_label = f"Tmean Anomaly"# (Slope={Tmean_slope:.3f}°C/dec, p={Tmean_p:.3f})"
    ax.plot(merged["year"], merged["Tmax_anomaly"], color="red", linewidth=1.8, label=Tmax_label)
    ax.plot(merged["year"], merged["Tmin_anomaly"], color="blue", linewidth=1.8, label=Tmin_label)
    ax.plot(merged["year"], merged["Tmean_anomaly"], color="black", linewidth=2, label=Tmean_label)
    # Axis styling
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{zone}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Std. Anomaly[°C]", fontsize=10, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    # Bold y-axis ticks
    ax.tick_params(axis='y', labelsize=9)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    # X label only for last subplot
    if i == len(stations) - 1:
        ax.set_xlabel("Year", fontsize=11, fontweight="bold")
    # Bold legend text
    leg = ax.legend(loc="upper left", fontsize=8, frameon=True)
    for text in leg.get_texts():
        text.set_fontweight("bold")
# === 5. FINAL LAYOUT ===
#fig.suptitle("STANDARDIZED TEMPERATURE ANOMALIES (TMAX, TMIN & TMEAN) BY ZONE WITH TRENDS (°C/Decade)",
             #fontsize=16, fontweight="bold", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("Standardized_Temperature_Anomalies_Trends_Zones.png", bbox_inches='tight')
plt.show()


# %% ==========================================================================
# 2. HEATWAVE DETECTION USING THE EXCESS HEAT FACTOR (EHF)
#    Paper: Section 3.2  --  Heatwave episodes, duration and frequency
#    Output: station_heatwave_events.csv
#            Figs. 3-7  --  events_<zone>.png
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

# ---------------- CONFIG ----------------
DATA_DIR = r"C:\Users\ampon\Desktop\PYTHON\EHFdata\EHFdata"
TMAX_PAT = "{name}_1960_2022_dTx_complete.txt"
TMIN_PAT = "{name}_1960_2022_dTn_complete.txt"
REF_PERIOD = (1961, 2022)          # for pooled T95
X_MIN, X_MAX = 1960, 2022

ZONES = {
    "Sudan Savannah":  ["Navrongo", "Zuarungu"],
    "Guinea Savannah": ["Wa", "Bole", "Tamale", "Yendi", "Kete_Krachi"],
    "Transition":      ["Sunyani", "Wenchi"],
    "Forest":          ["Ho", "Kumasi", "Akim_Oda", "Sefwi_Bekwai"],
    "Coastal":         ["Accra", "Akatsi", "Tema"],
}
INTENS = "mean_intensity"          # bar colour metric ("mean_intensity" or "peak_intensity")

# ---------------- EHF PIPELINE ----------------
def station_tmean(name):
    fx = os.path.join(DATA_DIR, TMAX_PAT.format(name=name))
    fn = os.path.join(DATA_DIR, TMIN_PAT.format(name=name))
    Tx = pd.read_csv(fx, sep=r"\s+", header=None, names=["year","month","day","Tmax"])
    Tn = pd.read_csv(fn, sep=r"\s+", header=None, names=["year","month","day","Tmin"])
    df = pd.merge(Tx, Tn, on=["year","month","day"])
    df["date"] = pd.to_datetime(df[["year","month","day"]])
    df = df.sort_values("date").set_index("date")
    tmean = (df["Tmax"] + df["Tmin"]) / 2.0
    idx = pd.date_range(tmean.index.min(), tmean.index.max(), freq="D")
    return tmean.reindex(idx)

def compute_ehf(tmean):
    r0, r1 = REF_PERIOD
    ref = (tmean.index.year >= r0) & (tmean.index.year <= r1)
    T95 = np.nanpercentile(tmean[ref].values, 95)
    tdp3 = tmean.rolling(3, min_periods=3).mean()              # days i-2,i-1,i
    t30  = tmean.shift(3).rolling(30, min_periods=30).mean()   # days i-3..i-32
    ehi_sig, ehi_accl = tdp3 - T95, tdp3 - t30
    return ehi_sig * ehi_accl.where(ehi_accl > 1.0, 1.0)

def detect_events(ehf):
    pos = ehf.dropna(); pos = pos[pos > 0]
    rows = []
    if not pos.empty:
        breaks = pos.index.to_series().diff().dt.days.ne(1).cumsum()
        for _, idx in pos.groupby(breaks):
            if len(idx) >= 3:
                v = idx.values
                rows.append({"year": idx.index[0].year,
                             "doy_start": idx.index[0].dayofyear,
                             "duration": len(idx),
                             "mean_intensity": float(v.mean()),
                             "peak_intensity": float(v.max())})
    return pd.DataFrame(rows, columns=["year","doy_start","duration",
                                       "mean_intensity","peak_intensity"])

# ---------------- BUILD EVENTS FOR ALL STATIONS ----------------
all_rows = []
for zone, stations in ZONES.items():
    for st in stations:
        try:
            ev = detect_events(compute_ehf(station_tmean(st)))
        except FileNotFoundError:
            print(f"[warn] missing file(s) for {st} — skipped"); continue
        ev.insert(0, "station", st); ev.insert(0, "zone", zone)
        all_rows.append(ev)
events = pd.concat(all_rows, ignore_index=True)
events.to_csv(os.path.join(DATA_DIR, "station_heatwave_events.csv"), index=False)
print(events.groupby(["zone","station"]).size())

# ---------------- PLOT: one figure per zone, one panel per station ----------------
cmap = LinearSegmentedColormap.from_list("ehf", ["yellow","orange","red"])
xticks = np.arange(X_MIN, X_MAX + 1, 2)

def plot_zone(zone, stations, outname):
    sub_all = events[events["zone"] == zone]
    norm = Normalize(vmin=sub_all[INTENS].min(), vmax=sub_all[INTENS].max())  # shared within zone
    n = len(stations)
    fig, axes = plt.subplots(n, 1, figsize=(12, 4.2 * n), sharex=False)
    axes = np.atleast_1d(axes)
    for ax, st in zip(axes, stations):
        s = events[(events["zone"] == zone) & (events["station"] == st)]
        for ep in s.itertuples():
            ax.bar(ep.year, ep.duration, bottom=ep.doy_start, width=0.6,
                   color=cmap(norm(getattr(ep, INTENS))), edgecolor="black")
            ax.text(ep.year - 0.28, ep.doy_start + ep.duration/2, f"{ep.duration}",
                    ha="right", va="center", fontsize=8)
        ax.set_xlim(X_MIN - 0.5, X_MAX + 0.5)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticks, rotation=90, fontweight="bold", fontsize=9)
        ax.set_yticks(np.arange(0, 366, 30))
        ax.set_yticklabels([str(dd) for dd in range(1, 366, 30)], fontweight="bold")
        ax.set_ylabel("Day of Year", fontsize=11, fontweight="bold")
        ax.set_title(f"{st.replace('_',' ')}  ({zone})", fontsize=12, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.35)
    sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=list(axes), orientation="vertical", fraction=0.025, pad=0.015)
    cbar.set_label("Mean EHF Intensity (°C²)", fontsize=12, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 0.85, 0.97])
    plt.savefig(os.path.join(DATA_DIR, outname), bbox_inches="tight"); plt.show(); plt.close()

for zone, stations in ZONES.items():
    plot_zone(zone, stations, f"events_{zone.replace(' ','_')}.png")


# %% ==========================================================================
# 3. TOTAL HEATWAVE DAYS AND ZONAL TRENDS
#    Paper: Section 3.3  --  Total Heatwave days and trends
#    Output: Fig. 8  --  5-roll.png
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
zones = ["Sudan Savannah", "Guinea Savannah", "Transition", "Forest", "Coastal"]
# --- Build total heatwave days per zone-year from the events file ---
events = pd.read_csv("heatwave_events.csv", parse_dates=["start_date", "end_date"])
years = range(1961, 2023)
hw_days = (events.groupby(["zone", "year"])["duration"].sum()
                 .reset_index(name="total_hw_days"))
full = pd.MultiIndex.from_product([zones, years], names=["zone", "year"])
hw_days = (hw_days.set_index(["zone", "year"])
                  .reindex(full, fill_value=0)         # years with no events -> 0
                  .reset_index())
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 10))
axes = axes.flatten()
for ax, zone in zip(axes, zones):
    zd = hw_days[hw_days["zone"] == zone].sort_values("year").copy()
    sns.scatterplot(data=zd, x="year", y="total_hw_days",
                    color="coral", ax=ax, s=30, alpha=0.7)
    zd["rolling_5yr"] = zd["total_hw_days"].rolling(window=5, center=True).mean()
    ax.plot(zd["year"], zd["rolling_5yr"], color="darkred", linewidth=2,
            label="5-year rolling mean")
    # Rate from the RAW annual totals (this is the number you report), with p-value
    slope, intercept, r, p, se = linregress(zd["year"], zd["total_hw_days"])
    ax.text(0.02, 0.92, f"Rate = {slope:.2f} days/yr (p = {p:.3f})",
            transform=ax.transAxes, fontsize=9, fontweight="bold",
            ha="left", va="top")
    ax.set_title(zone, fontsize=12, fontweight="bold", loc="left")
    ax.tick_params(axis="x", labelrotation=90)
    ax.set_xlabel("")
    ax.set_ylabel("Total Heatwave Days")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=8, loc="upper center")
for i in range(len(zones), len(axes)):
    fig.delaxes(axes[i])
fig.suptitle("EHF-Based Annual Total Heatwave Days with 5-Year Rolling Mean (1961–2022)",
             fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("5-roll.png", bbox_inches="tight")
plt.show()


# %% ==========================================================================
# 4. STATION-LEVEL TRENDS IN ANNUAL TOTAL HEATWAVE DAYS
#    Paper: Section 3.3  --  Table 3
#    Output: station_hwday_trends.csv  (plus LaTeX table rows to console)
# =============================================================================

import os
import pandas as pd
import numpy as np
from scipy.stats import linregress
DATA_DIR = r"C:\Users\ampon\Desktop\PYTHON\EHFdata\EHFdata"
YEARS = range(1961, 2023)
ZONES = {
    "Sudan Savannah":  ["Navrongo", "Zuarungu"],
    "Guinea Savannah": ["Wa", "Bole", "Tamale", "Yendi", "Kete_Krachi"],
    "Transition":      ["Sunyani", "Wenchi"],
    "Forest":          ["Ho", "Kumasi", "Akim_Oda", "Sefwi_Bekwai"],
    "Coastal":         ["Accra", "Akatsi", "Tema"],
}
events = pd.read_csv(os.path.join(DATA_DIR, "station_heatwave_events.csv"))
hw = (events.groupby(["zone", "station", "year"])["duration"]
            .sum().reset_index(name="total_hw_days"))
rows = []
for zone, stations in ZONES.items():
    for st in stations:
        sd = hw[(hw["zone"] == zone) & (hw["station"] == st)]
        sd = (sd.set_index("year").reindex(YEARS, fill_value=0)
                .rename_axis("year").reset_index())
        slope, intercept, r, p, se = linregress(sd["year"], sd["total_hw_days"])
        rows.append({"Zone": zone, "Station": st.replace("_", " "),
                     "slope": slope, "p": p})
tbl = pd.DataFrame(rows)
tbl.round(3).to_csv(os.path.join(DATA_DIR, "station_hwday_trends.csv"), index=False)
def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
# console check
for _, r in tbl.iterrows():
    print(f"{r['Zone']:16s} {r['Station']:14s} {r['slope']:.2f} (p={r['p']:.3f}){stars(r['p'])}")
# ---- LaTeX rows with merged zone cells ----
print("\n% --- paste into table body ---")
for zone, stations in ZONES.items():
    sub = tbl[tbl["Zone"] == zone]
    n = len(sub)
    for j, (_, r) in enumerate(sub.iterrows()):
        pstr = "$<$0.001" if r["p"] < 0.001 else f"{r['p']:.3f}"
        cell = f"{r['slope']:.2f} ({pstr})$^{{{stars(r['p'])}}}$" if stars(r['p']) else f"{r['slope']:.2f} ({pstr})"
        if j == 0:
            print(f"\\multirow{{{n}}}{{*}}{{{zone}}}")
            print(f" & {r['Station']:12s} & {cell} \\\\")
        else:
            print(f" & {r['Station']:12s} & {cell} \\\\")
    print("\\hline")


# %% ==========================================================================
# 5. RETURN LEVELS OF EHF INTENSITY AND HEATWAVE DURATION
#    Paper: Section 3.4  --  Return Levels of Heatwave Intensity
#    Method: Peaks-Over-Threshold with the Generalized Pareto Distribution;
#            95% CIs from parametric bootstrap.
#    Output: Fig. 9  --  FigS_station_return_levels.png
# =============================================================================

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import genpareto
from scipy.optimize import minimize
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DATA_DIR = r"C:\Users\ampon\Desktop\PYTHON\EHFdata\EHFdata"
TMAX_PAT = "{name}_1960_2022_dTx_complete.txt"
TMIN_PAT = "{name}_1960_2022_dTn_complete.txt"
REF_PERIOD = (1961, 2022); N_YEARS = REF_PERIOD[1]-REF_PERIOD[0]+1
RETURN_PERIODS = [2, 5, 10, 20, 50]
SHAPE_BOUNDS = (-0.5, 0.5)
POT_INT_U, POT_DUR_U = 0.0, 2.5
N_BOOT, SEED = 500, 42

ZONES = {
    "Sudan Savannah":  ["Navrongo", "Zuarungu"],
    "Guinea Savannah": ["Wa", "Bole", "Tamale", "Yendi", "Kete_Krachi"],
    "Transition":      ["Sunyani", "Wenchi"],
    "Forest":          ["Ho", "Kumasi", "Akim_Oda", "Sefwi_Bekwai"],
    "Coastal":         ["Accra", "Akatsi", "Tema"],
}

# ---------------- EHF + EVENTS ----------------
def station_tmean(name):
    fx = os.path.join(DATA_DIR, TMAX_PAT.format(name=name))
    fn = os.path.join(DATA_DIR, TMIN_PAT.format(name=name))
    Tx = pd.read_csv(fx, sep=r"\s+", header=None, names=["year","month","day","Tmax"])
    Tn = pd.read_csv(fn, sep=r"\s+", header=None, names=["year","month","day","Tmin"])
    df = pd.merge(Tx, Tn, on=["year","month","day"])
    df["date"] = pd.to_datetime(df[["year","month","day"]])
    df = df.sort_values("date").set_index("date")
    t = (df["Tmax"] + df["Tmin"]) / 2.0
    return t.reindex(pd.date_range(t.index.min(), t.index.max(), freq="D"))

def compute_ehf(tmean):
    r0, r1 = REF_PERIOD
    ref = (tmean.index.year >= r0) & (tmean.index.year <= r1)
    T95 = np.nanpercentile(tmean[ref].values, 95)
    tdp3 = tmean.rolling(3, min_periods=3).mean()
    t30  = tmean.shift(3).rolling(30, min_periods=30).mean()
    sig, accl = tdp3 - T95, tdp3 - t30
    return sig * accl.where(accl > 1.0, 1.0)

def event_stats(ehf):
    pos = ehf.dropna(); pos = pos[pos > 0]; peaks, durs = [], []
    if not pos.empty:
        br = pos.index.to_series().diff().dt.days.ne(1).cumsum()
        for _, idx in pos.groupby(br):
            if len(idx) >= 3:
                peaks.append(float(idx.values.max())); durs.append(len(idx))
    return np.array(peaks, float), np.array(durs, float)

# ---------------- GPD / POT + bootstrap CI ----------------
def _nll(p, exc):
    c, s = p
    if s <= 0: return np.inf
    ll = genpareto.logpdf(exc, c, loc=0, scale=s)
    return np.inf if not np.all(np.isfinite(ll)) else -np.sum(ll)

def fit_gpd(exc):
    r = minimize(_nll, [0.1, max(np.mean(exc),1e-3)], args=(exc,),
                 method="L-BFGS-B", bounds=[SHAPE_BOUNDS, (1e-6, None)])
    return r.x

def rl_from(c, s, u, rate, Ts):
    out = []
    for T in Ts:
        m = rate*T
        out.append(u + genpareto.ppf(1-1/m, c, loc=0, scale=s) if m > 1 else np.nan)
    return np.array(out)

def pot_curve(values, u, Ts):
    """Return (point estimate, lo, hi) curves, or None if too few events."""
    exc = values[values > u] - u
    if exc.size < 10:
        return None
    c, s = fit_gpd(exc); rate = exc.size / N_YEARS
    point = rl_from(c, s, u, rate, Ts)
    rng = np.random.default_rng(SEED)
    boot = []
    for _ in range(N_BOOT):
        samp = genpareto.rvs(c, loc=0, scale=s, size=exc.size, random_state=rng)
        bc, bs = fit_gpd(samp)
        boot.append(rl_from(bc, bs, u, rate, Ts))
    boot = np.array(boot)
    lo = np.nanpercentile(boot, 2.5, axis=0)
    hi = np.nanpercentile(boot, 97.5, axis=0)
    return point, lo, hi

# ---------------- build ----------------
records = {}
for zone, stations in ZONES.items():
    for st in stations:
        try:
            ehf = compute_ehf(station_tmean(st))
        except FileNotFoundError:
            print(f"[warn] missing files for {st}"); continue
        peaks, durs = event_stats(ehf)
        records[(zone, st)] = {"int": pot_curve(peaks, POT_INT_U, RETURN_PERIODS),
                               "dur": pot_curve(durs,  POT_DUR_U, RETURN_PERIODS)}

# ---------------- PLOT ----------------
zones = list(ZONES.keys()); Ts = RETURN_PERIODS
fig, axes = plt.subplots(len(zones), 2, figsize=(10, 2.6*len(zones)))
fig.subplots_adjust(left=0.13, hspace=0.42, wspace=0.22)

def draw(ax, zone, key, ylabel, first):
    stations = ZONES[zone]
    colors = plt.cm.viridis(np.linspace(0, 0.85, len(stations)))
    for st, col in zip(stations, colors):
        rec = records.get((zone, st))
        if rec is None or rec[key] is None:
            continue
        pt, lo, hi = rec[key]
        ax.plot(Ts, pt, "-o", color=col, lw=1.6, ms=3.5, label=st.replace("_"," "))
        ax.fill_between(Ts, lo, hi, color=col, alpha=0.13)
    ax.set_xscale("log"); ax.set_xticks(Ts)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    # ax.grid(True, which="both", alpha=0.3)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=7, frameon=False)

for i, zone in enumerate(zones):
    draw(axes[i,0], zone, "int", "EHF intensity (°C$^2$)", i==0)
    draw(axes[i,1], zone, "dur", "Duration (days)", i==0)
    # zone label, pulled in close to the left panel
    axes[i,0].annotate(zone, xy=(-0.22, 0.5), xycoords="axes fraction",
                       rotation=90, va="center", ha="center",
                       fontsize=11, fontweight="bold")
    if i == 0:
        axes[i,0].set_title("Intensity return levels", fontsize=12, fontweight="bold")
        axes[i,1].set_title("Duration return levels", fontsize=12, fontweight="bold")
    if i == len(zones)-1:
        axes[i,0].set_xlabel("Return period (years)", fontsize=10)
        axes[i,1].set_xlabel("Return period (years)", fontsize=10)

plt.savefig(os.path.join(DATA_DIR, "FigS_station_return_levels.png"),
            bbox_inches="tight", dpi=200)
plt.show(); plt.close()